In [13]:
from qiskit_ibm_runtime import QiskitRuntimeService

service = QiskitRuntimeService(channel="ibm_quantum_platform", token="2q_XdIifzC1oIntav5ojfZXAiie_zVxQ1SppfAP_aIAN")



qiskit_runtime_service._resolve_cloud_instances:WARNING:2025-09-06 00:37:51,979: Default instance not set. Searching all available instances.


In [14]:
QiskitRuntimeService.save_account(channel="ibm_quantum_platform",token="2q_XdIifzC1oIntav5ojfZXAiie_zVxQ1SppfAP_aIAN")

In [16]:
backend = service.backend(name="ibm_brisbane")

backend.num_qubits

127

In [11]:
import numpy as np
from qiskit import QuantumCircuit
from qiskit_aer import Aer

# Number of qubits (bits to exchange)
n = 20  

# Step 1: Alice generates random bits and bases
alice_bits = np.random.randint(2, size=n)
alice_bases = np.random.randint(2, size=n)

# Step 2: Eve generates random bases
eve_bases = np.random.randint(2, size=n)

# Step 3: Bob generates random bases
bob_bases = np.random.randint(2, size=n)

# Function to encode a qubit
def encode_qubit(bit, basis):
    qc = QuantumCircuit(1, 1)
    if bit == 1:
        qc.x(0)
    if basis == 1:
        qc.h(0)
    return qc

# Function to measure in given basis
def measure_qubit(qc, basis):
    if basis == 1:
        qc.h(0)
    qc.measure(0, 0)
    return qc

# Quantum simulator
simulator = Aer.get_backend("aer_simulator")

# Step 4: Eve intercepts and resends
eve_results = []
intercepted_qubits = []

for i in range(n):
    qc = encode_qubit(alice_bits[i], alice_bases[i])

    # Eve measures
    eve_qc = measure_qubit(qc.copy(), eve_bases[i])
    sim_job = simulator.run(eve_qc, shots=1)
    result = sim_job.result()
    counts = result.get_counts()
    eve_bit = int(max(counts, key=counts.get))
    eve_results.append(eve_bit)

    # Eve resends to Bob
    resent_qc = encode_qubit(eve_bit, eve_bases[i])
    intercepted_qubits.append(resent_qc)

# Step 5: Bob measures the intercepted qubits
bob_results = []
for i in range(n):
    qc = measure_qubit(intercepted_qubits[i], bob_bases[i])
    sim_job = simulator.run(qc, shots=1)
    result = sim_job.result()
    counts = result.get_counts()
    measured_bit = int(max(counts, key=counts.get))
    bob_results.append(measured_bit)

# Step 6: Sifting
sifted_key_alice = []
sifted_key_bob = []

for i in range(n):
    if alice_bases[i] == bob_bases[i]:
        sifted_key_alice.append(alice_bits[i])
        sifted_key_bob.append(bob_results[i])

# Step 7: Error rate estimation
errors = sum([1 for a, b in zip(sifted_key_alice, sifted_key_bob) if a != b])
error_rate = errors / len(sifted_key_alice) if sifted_key_alice else 0

print("Alice bits:    ", alice_bits)
print("Alice bases:   ", alice_bases)
print("Eve bases:     ", eve_bases)
print("Eve results:   ", eve_results)
print("Bob bases:     ", bob_bases)
print("Bob results:   ", bob_results)
print("Sifted Alice:  ", sifted_key_alice)
print("Sifted Bob:    ", sifted_key_bob)
print("Error rate:    {:.2f}".format(error_rate))


Alice bits:     [1 1 0 0 0 0 1 1 0 1 0 0 1 0 0 0 1 0 1 0]
Alice bases:    [0 1 1 1 0 1 1 1 0 0 0 1 1 0 0 0 0 0 1 0]
Eve bases:      [0 1 0 1 1 0 1 0 0 0 0 0 0 1 0 1 1 0 1 1]
Eve results:    [1, 1, 1, 0, 1, 0, 1, 1, 0, 1, 0, 0, 1, 1, 0, 0, 0, 0, 1, 1]
Bob bases:      [0 0 1 0 0 0 1 0 1 0 0 0 0 0 0 1 0 1 0 0]
Bob results:    [1, 1, 1, 1, 0, 0, 1, 1, 0, 1, 0, 0, 1, 0, 0, 0, 0, 1, 1, 0]
Sifted Alice:   [np.int32(1), np.int32(0), np.int32(0), np.int32(1), np.int32(1), np.int32(0), np.int32(0), np.int32(0), np.int32(1), np.int32(0)]
Sifted Bob:     [1, 1, 0, 1, 1, 0, 0, 0, 0, 0]
Error rate:    0.20
